# Test Notebook: Open Library Public Domain Opportunity Radar

This notebook checks `helper_code.py` without the real dumps. Every test
builds a small input whose correct answer we can work out by hand, runs a
helper function on it, and compares the result with that answer.

The tests are grouped by what could go wrong:

1. **Rule functions (pure Python).** Year parsing, publisher names and
   the copyright rules, including every boundary year.
2. **Spark twins.** Each rule exists twice: a readable Python version and
   a Spark column expression used on the full data. We run both on the
   same inputs and require identical answers.
3. **End-to-end on a miniature dump.** We write tiny `.txt.gz` files in
   the real Open Library format (including a malformed record and a
   deprecated Wikidata value; the run without Wikidata, as on Jojie, is
   tested too), run the whole pipeline, and check every
   number against hand calculations.
4. **Style.** `pycodestyle` on the helper module.
5. **Regimes, audit and diagram.** The three-regime table, the
   hand-checked audit with one planted error, and the pipeline diagram.
6. **Locating the dumps.** The fixed snapshot folder on Jojie and the
   file-name rules.
7. **Whole-snapshot files.** The revision history and the all-types
   dump, including the history of an author's death year.
8. **Exploration helpers.** Raw-data statistics, the missing-value
   breakdowns, the snapshot inventory and the watchlist review.
9. **Merged records.** Redirect chains, merged keys, duplicate titles
   and the two sensitivity checks.

Run it from top to bottom (Kernel > Restart & Run All). A summary at the
end says how many tests passed. The tests write only to a temporary
folder, which is deleted afterwards.

In [1]:
from pathlib import Path
import gzip
import importlib
import json
import os
import sys
import tempfile
import unittest

import matplotlib.pyplot as plt
import pandas as pd

sys.path.insert(0, str(Path.cwd()))
import helper_code as helper

helper = importlib.reload(helper)

# A small local Spark session is enough for the miniature data.
spark = helper.create_spark(app_name="ol-pd-radar-tests",
                            driver_memory="2g", shuffle_partitions=4)
print("Spark", spark.version)

Spark 4.1.1


## 1 Rule functions

These tests pin the behaviour at the boundaries, where off-by-one
mistakes happen. On **January 1, 2027** a US work published in 1931
reaches the end of its 95-year term (1931 + 95 = 2026, protected until
December 31, 2026). In the Philippines, an author who died in 1976 has
been protected for 50 years counted from January 1, 1977, which ends on
December 31, 2026.

In [2]:
class TestYearParsing(unittest.TestCase):
    # Free-text publication dates -> first plausible year.

    def test_publication_years(self):
        cases = {
            "February 27, 1997": 1997,
            "c1931": 1931,
            "1931-1932": 1931,          # the first year is kept
            "Printed 1931; reprinted 1950": 1931,
            "19--": None,               # an unknown decade is not a year
            "1200": None,               # before printing (MIN_VALID_YEAR)
            "2030": None,               # after the dump date
            "12345": None,              # five digits are not a year
            "": None,
            None: None,
        }
        for text, expected in cases.items():
            with self.subTest(text=text):
                self.assertEqual(helper.parse_year(text), expected)

    def test_life_years(self):
        # Author dates keep the LATEST year (conservative) and handle BC.
        cases = {
            "1901.7.": 1901,
            "1963)": 1963,
            "October 31, 1987": 1987,
            "c. 1850 or 1851": 1851,
            "1016": 1016,
            "384 BC": -384,
            "384 B.C.": -384,
            "unknown": None,
            None: None,
        }
        for text, expected in cases.items():
            with self.subTest(text=text):
                self.assertEqual(helper.parse_life_year(text), expected)

    def test_wikidata_time(self):
        self.assertEqual(
            helper.parse_wikidata_time_year("+1931-05-03T00:00:00Z"), 1931)
        self.assertEqual(
            helper.parse_wikidata_time_year("-0384-00-00T00:00:00Z"), -384)
        self.assertIsNone(helper.parse_wikidata_time_year("garbage"))


class TestPublisherNames(unittest.TestCase):

    def test_normalization(self):
        cases = {
            "Penguin Books Ltd.": "penguin books",
            "PENGUIN BOOKS": "penguin books",
            "Hal Leonard Publishing Corporation": "hal leonard publishing",
            "Simon & Schuster, Inc.": "simon schuster",
            "  ": "",
            None: "",
        }
        for name, expected in cases.items():
            with self.subTest(name=name):
                self.assertEqual(helper.normalize_publisher(name), expected)


class TestCopyrightRules(unittest.TestCase):

    def test_us_boundaries(self):
        cases = {1500: "pd", 1930: "pd", 1931: "enters", 1932: "renewal",
                 1963: "renewal", 1964: "protected_pre78",
                 1977: "protected_pre78", 1978: "protected_post78",
                 2026: "protected_post78", None: "unknown"}
        for year, expected in cases.items():
            with self.subTest(year=year):
                self.assertEqual(helper.us_status(year), expected)

    def test_us_rule_moves_with_reference_year(self):
        # On Jan 1, 2028 the 1932 works enter and 1931 works are PD.
        self.assertEqual(helper.us_status(1932, reference_year=2028),
                         "enters")
        self.assertEqual(helper.us_status(1931, reference_year=2028), "pd")

    def test_philippine_life_plus_50(self):
        rule = helper.life_plus_status
        self.assertEqual(rule(1975, 1, 1, 50), "pd")
        self.assertEqual(rule(1976, 1, 1, 50), "enters")
        self.assertEqual(rule(1977, 1, 1, 50), "protected")
        self.assertEqual(rule(-384, 1, 1, 50), "pd")
        # two authors, only one death year known -> cannot decide
        self.assertEqual(rule(1900, 2, 1, 50), "unknown_death")
        self.assertEqual(rule(None, 1, 0, 50), "unknown_death")
        self.assertEqual(rule(None, 0, 0, 50), "unknown_author")

    def test_life_plus_70(self):
        rule = helper.life_plus_status
        self.assertEqual(rule(1955, 1, 1, 70), "pd")
        self.assertEqual(rule(1956, 1, 1, 70), "enters")
        self.assertEqual(rule(1957, 1, 1, 70), "protected")

    def test_death_year_source(self):
        self.assertEqual(helper.choose_death_year(1951, 1950),
                         (1950, "wikidata"))
        self.assertEqual(helper.choose_death_year(1951, None),
                         (1951, "open_library"))
        self.assertEqual(helper.choose_death_year(None, None), (None, None))

## 2 Spark expressions agree with the Python rules

The full-data pipeline cannot call the Python functions row by row
(that would be very slow), so it uses equivalent Spark expressions. If
the two versions ever disagreed, the report's numbers would silently
differ from the documented rules. We therefore evaluate both on the same
inputs, including every year from 1450 to 2026.

In [3]:
from pyspark.sql import functions as F


class TestSparkTwins(unittest.TestCase):

    def test_year_from_text(self):
        texts = ["February 27, 1997", "c1931", "19--", "1200", "2030",
                 "12345", "1931-1932", "Printed 1931; reprinted 1950", None]
        df = spark.createDataFrame([(t,) for t in texts], "t string")
        got = [r[0] for r in df.select(
            helper.year_from_text(F.col("t"))).collect()]
        self.assertEqual(got, [helper.parse_year(t) for t in texts])

    def test_life_year_from_text(self):
        texts = ["1901.7.", "1963)", "c. 1850 or 1851", "384 BC",
                 "384 B.C.", "unknown", "1016", None]
        df = spark.createDataFrame([(t,) for t in texts], "t string")
        got = [r[0] for r in df.select(
            helper.life_year_from_text(F.col("t"))).collect()]
        self.assertEqual(got, [helper.parse_life_year(t) for t in texts])

    def test_publisher_key(self):
        names = ["Penguin Books Ltd.", "Hal Leonard Publishing Corporation",
                 "Simon & Schuster, Inc.", "  ", "Ateneo de Manila Univ."]
        df = spark.createDataFrame([(n,) for n in names], "n string")
        got = [r[0] for r in df.select(
            helper.publisher_key(F.col("n"))).collect()]
        self.assertEqual(got, [helper.normalize_publisher(n) for n in names])

    def test_us_status_every_year(self):
        years = list(range(1450, 2027)) + [None]
        df = spark.createDataFrame([(y,) for y in years], "y int")
        got = [r[0] for r in df.select(
            helper.us_status_column(F.col("y"))).collect()]
        self.assertEqual(got, [helper.us_status(y) for y in years])

    def test_life_plus_every_case(self):
        rows = [(d, n, k) for d in [None, -384, 1900, 1955, 1956, 1957,
                                    1975, 1976, 1977, 2020]
                for n in [0, 1, 2] for k in [0, 1, 2] if k <= n]
        df = spark.createDataFrame(rows, "d int, n int, k int")
        for term in (50, 70):
            got = [r[0] for r in df.select(helper.life_plus_status_column(
                F.col("d"), F.col("n"), F.col("k"), term)).collect()]
            expected = [helper.life_plus_status(d, n, k, term)
                        for d, n, k in rows]
            self.assertEqual(got, expected, msg=f"term={term}")

## 3 End-to-end on a miniature dump

The fixture below imitates the real files closely: tab-separated lines,
JSON records, `\N` for missing values, and Wikidata JSON wrapped in
CSV-style quotes. It contains deliberate traps, listed next to each
record. The expected values in the tests were worked out by hand from
these records, not by running the code.

| Work | Authors (death year) | Dates | Expected US / PH |
|---|---|---|---|
| W1 | A3 (unknown) | first published 1931 | enters 2027 / unknown |
| W2 | A1 (1976), A2 (1950) | first published 1960, an edition from 1965 | renewal unknown / enters 2027 |
| W3 | A2 (1950) | editions 1890 and 2020 | public domain / public domain |
| W4 | A4 (1970, Filipino) | first published 1955 | renewal unknown / public domain |
| W5 | none | none | unknown / no author |
| W6 | A3 (born 1900) | "first published" 1800 | flagged: year before birth |
| W7 | — | malformed JSON | kept, json_ok = False |

In [4]:
def dump_line(record_type, key, record):
    """One line in the works/editions/authors dump format.

    Real records repeat their key and type inside the JSON; the parser
    relies on that key to tell parsed from malformed records.
    """
    record = {"key": key, "type": {"key": record_type}, **record}
    return "\t".join([record_type, key, "1", "2022-09-01T00:00:00",
                      json.dumps(record)])


def wikidata_line(qid, statements):
    """One line in the wikidata dump format (CSV-quoted JSON)."""
    entity = json.dumps({"id": qid, "type": "item",
                         "statements": statements})
    return "\t".join([qid, '"' + entity.replace('"', '""') + '"',
                      "2022-09-18 00:00:00"])


def time_statement(year, precision=11, rank="normal"):
    return {"rank": rank, "value": {"content": {
        "time": f"+{year:04d}-01-01T00:00:00Z", "precision": precision}}}


def id_statement(value, rank="normal"):
    return {"rank": rank, "value": {"content": value}}


def author_ref(key):
    return {"type": {"key": "/type/author_role"},
            "author": {"key": f"/authors/{key}"}}


works = [
    dump_line("/type/work", "/works/OL1W", {
        "title": "Book Nineteen Thirty-One", "first_publish_date": "1931",
        "authors": [author_ref("OL3A")], "subjects": ["Fiction"]}),
    dump_line("/type/work", "/works/OL2W", {
        "title": "Two Authors", "first_publish_date": "May 1960",
        "authors": [author_ref("OL1A"), author_ref("OL2A")]}),
    dump_line("/type/work", "/works/OL3W", {
        "title": "Old Classic", "authors": [author_ref("OL2A")]}),
    dump_line("/type/work", "/works/OL4W", {
        "title": "Filipino Novel", "first_publish_date": "1955",
        "authors": [author_ref("OL4A")]}),
    dump_line("/type/work", "/works/OL5W", {"title": "No Author"}),
    dump_line("/type/work", "/works/OL6W", {
        "title": "Impossible Date", "first_publish_date": "1800",
        "authors": [author_ref("OL3A")]}),
    "/type/work\t/works/OL7W\t1\t2022-09-01T00:00:00\t{not valid json",
    # a redirect record must be ignored
    dump_line("/type/redirect", "/works/OL8W", {"location": "/works/OL1W"}),
]

editions = [
    dump_line("/type/edition", "/books/OL1M", {
        "works": [{"key": "/works/OL1W"}], "publish_date": "1931",
        "publishers": ["Old Press Ltd."],
        "languages": [{"key": "/languages/eng"}]}),
    dump_line("/type/edition", "/books/OL2M", {
        "works": [{"key": "/works/OL1W"}], "publish_date": "2020",
        "publishers": ["Old Press"], "ocaid": "oldbook00",
        "languages": [{"key": "/languages/tgl"}]}),
    dump_line("/type/edition", "/books/OL3M", {
        "works": [{"key": "/works/OL3W"}], "publish_date": "1890",
        "publishers": ["Alpha"]}),
    dump_line("/type/edition", "/books/OL4M", {
        "works": [{"key": "/works/OL3W"}], "publish_date": "March 2020",
        "publishers": ["Beta Inc."],
        "languages": [{"key": "/languages/fil"}]}),
    dump_line("/type/edition", "/books/OL5M", {
        "works": [{"key": "/works/OL2W"}], "publish_date": "1965"}),
    # an edition with no work, and one pointing to a missing work
    dump_line("/type/edition", "/books/OL6M", {"publish_date": "2019"}),
    dump_line("/type/edition", "/books/OL7M", {
        "works": [{"key": "/works/OL999W"}], "publish_date": "2018"}),
]

authors = [
    dump_line("/type/author", "/authors/OL1A", {
        "name": "Died Seventy-Six", "death_date": "1976"}),
    # Open Library says 1951 but Wikidata says 1950: Wikidata wins
    dump_line("/type/author", "/authors/OL2A", {
        "name": "Mid Century", "death_date": "1951"}),
    dump_line("/type/author", "/authors/OL3A", {
        "name": "Living Or Unknown", "birth_date": "1900"}),
    # linked to Wikidata only through remote_ids
    dump_line("/type/author", "/authors/OL4A", {
        "name": "Filipino Author", "remote_ids": {"wikidata": "Q4"}}),
]

wikidata = [
    wikidata_line("Q2", {
        "P648": [id_statement("OL2A"), id_statement("OL2W")],
        "P570": [time_statement(1950),
                 time_statement(1999, rank="deprecated"),  # ignored
                 time_statement(2000, precision=8)],       # decade only
        "P569": [time_statement(1880)],
    }),
    wikidata_line("Q4", {
        "P570": [time_statement(1970)],
        "P27": [id_statement("Q928")],                     # Philippines
    }),
]

reading_log = [
    "/works/OL1W\t\\N\tWant to Read\t2022-09-01",
    "/works/OL1W\t/books/OL1M\tWant to Read\t2021-12-01",
    "/works/OL1W\t\\N\tWant to Read\t2020-06-01",
    "/works/OL1W\t\\N\tAlready Read\t2022-01-15",
    "/works/OL3W\t\\N\tWant to Read\t2022-01-01",
    "/works/OL2W\t\\N\tCurrently Reading\t2022-02-01",
    "/works/OL1W\t\\N\tSome Custom Shelf\t2022-02-01",      # dropped
]

ratings = [
    "/works/OL1W\t\\N\t4\t2022-01-01",
    "/works/OL1W\t/books/OL2M\t5\t2022-02-01",
    "/works/OL3W\t\\N\t9\t2022-02-01",                       # invalid
]

tmp = tempfile.TemporaryDirectory()
fixture_dir = Path(tmp.name) / "dumps"
fixture_dir.mkdir()
for name, lines in {"works": works, "editions": editions,
                    "authors": authors, "wikidata": wikidata,
                    "reading-log": reading_log, "ratings": ratings}.items():
    path = fixture_dir / f"ol_dump_{name}_2022-09-30.txt.gz"
    with gzip.open(path, "wt", encoding="utf-8") as handle:
        handle.write("\n".join(lines) + "\n")

dumps = helper.discover_dumps(fixture_dir)
paths = helper.build_parquet_tables(spark, dumps, Path(tmp.name) / "pq")
tables = helper.load_tables(spark, paths)
deaths = helper.reconcile_author_deaths(tables["authors"],
                                        tables["wikidata"])
status = helper.estimate_copyright_status(
    tables["works"], tables["editions"], deaths["author_deaths"])
demand = helper.compute_demand(tables["reading_log"], tables["ratings"])
supply = helper.compute_supply(tables["editions"])
profile = helper.build_work_profile(status, demand, supply).cache()
rows = {r["work_key"]: r.asDict()
        for r in profile.collect()}
print("Fixture built:", {k: v.count() for k, v in tables.items()})

Decompressed 6 dumps in 0.0 min


Fixture built: {'works': 7, 'editions': 7, 'authors': 4, 'reading_log': 6, 'ratings': 3, 'wikidata': 2}


In [5]:
def work(n):
    return rows[f"/works/OL{n}W"]


class TestParsing(unittest.TestCase):

    def test_record_counts(self):
        # the redirect is dropped; the malformed work is kept
        self.assertEqual(tables["works"].count(), 7)
        self.assertEqual(tables["editions"].count(), 7)
        self.assertEqual(tables["authors"].count(), 4)
        self.assertEqual(tables["wikidata"].count(), 2)
        self.assertEqual(tables["reading_log"].count(), 6)
        self.assertEqual(tables["ratings"].count(), 3)

    def test_malformed_json_is_flagged_not_dropped(self):
        bad = tables["works"].filter("work_key = '/works/OL7W'").collect()
        self.assertEqual(len(bad), 1)
        self.assertFalse(bad[0]["json_ok"])

    def test_profile_counts_malformed_json(self):
        profile_table = helper.profile_tables(tables)
        parsed = profile_table.query(
            "table == 'works' and field == 'json parsed'").iloc[0]
        self.assertEqual((parsed["rows"], parsed["non_null"]), (7, 6))

    def test_invalid_rating_becomes_null(self):
        values = sorted(r["rating"] is None
                        for r in tables["ratings"].collect())
        self.assertEqual(values, [False, False, True])

    def test_parquet_reused_without_raw_dumps(self):
        # once tables exist, the raw dumps are no longer needed
        reused = helper.build_parquet_tables(spark, {}, Path(tmp.name) / "pq")
        # the optional Wikidata table is reused too, because it exists
        self.assertEqual(set(reused), set(helper.DUMP_NAMES) | {"wikidata"})
        # but a missing table with no dump is an error, not a silent skip
        with self.assertRaises(FileNotFoundError):
            helper.build_parquet_tables(spark, {}, Path(tmp.name) / "empty")
        self.assertEqual(
            helper.discover_dumps(Path(tmp.name), required=False), {})

    def test_no_duplicate_keys(self):
        dupes = helper.find_duplicate_keys(tables)
        self.assertEqual(dupes["duplicated_keys"].tolist(), [0, 0, 0])


class TestAuthorDeaths(unittest.TestCase):

    def setUp(self):
        self.by_key = {r["author_key"]: r for r in
                       deaths["author_deaths"].collect()}

    def test_wikidata_wins_and_deprecated_ignored(self):
        a2 = self.by_key["/authors/OL2A"]
        self.assertEqual(a2["death_year"], 1950)
        self.assertEqual(a2["death_source"], "wikidata")
        self.assertEqual(a2["birth_year"], 1880)

    def test_open_library_fallback(self):
        a1 = self.by_key["/authors/OL1A"]
        self.assertEqual((a1["death_year"], a1["death_source"]),
                         (1976, "open_library"))

    def test_link_through_remote_ids(self):
        a4 = self.by_key["/authors/OL4A"]
        self.assertEqual(a4["death_year"], 1970)
        self.assertTrue(a4["wd_philippine_citizen"])

    def test_without_wikidata(self):
        # the Jojie snapshot has no Wikidata dump: Open Library years only
        only_ol = helper.reconcile_author_deaths(tables["authors"])
        by_key = {r["author_key"]: r for r in
                  only_ol["author_deaths"].collect()}
        self.assertEqual((by_key["/authors/OL2A"]["death_year"],
                          by_key["/authors/OL2A"]["death_source"]),
                         (1951, "open_library"))
        self.assertIsNone(by_key["/authors/OL4A"]["death_year"])
        self.assertFalse(by_key["/authors/OL4A"]["wd_philippine_citizen"])
        self.assertEqual(
            int(only_ol["agreement"].iloc[0]["authors_with_both"]), 0)

    def test_agreement_table(self):
        agreement = deaths["agreement"].iloc[0]
        # only OL2A has both sources: 1951 vs 1950
        self.assertEqual(agreement["authors_with_both"], 1)
        self.assertEqual(agreement["exact_match"], 0)
        self.assertEqual(agreement["within_one_year"], 1)


class TestStatus(unittest.TestCase):

    def test_first_year(self):
        self.assertEqual(work(1)["first_year"], 1931)
        self.assertEqual(work(2)["first_year"], 1960)   # 1960 < 1965
        self.assertEqual(work(3)["first_year"], 1890)   # from editions
        self.assertEqual(work(3)["year_source"], "earliest edition")
        self.assertIsNone(work(5)["first_year"])

    def test_us_and_ph_codes(self):
        expected = {1: ("enters", "unknown_death"),
                    2: ("renewal", "enters"),
                    3: ("pd", "pd"),
                    4: ("renewal", "pd"),
                    5: ("unknown", "unknown_author")}
        for n, codes in expected.items():
            with self.subTest(work=n):
                self.assertEqual((work(n)["us_code"], work(n)["ph_code"]),
                                 codes)

    def test_last_author_counts(self):
        self.assertEqual(work(2)["n_authors"], 2)
        self.assertEqual(work(2)["last_death_year"], 1976)

    def test_flags(self):
        self.assertTrue(work(6)["year_before_birth"])
        self.assertTrue(work(6)["flagged"])
        self.assertFalse(work(6)["potentially_pd"])
        self.assertTrue(work(1)["enters_2027"])
        self.assertTrue(work(2)["enters_2027"])
        self.assertTrue(work(3)["potentially_pd"])
        self.assertTrue(work(4)["has_philippine_author"])

    def test_status_summary_adds_up(self):
        summary = helper.summarize_status(status)
        self.assertEqual(summary["us"]["works"].sum(), 7)
        self.assertEqual(summary["ph"]["works"].sum(), 7)
        self.assertEqual(int(summary["crosstab"].values.sum()), 7)


class TestDemandAndSupply(unittest.TestCase):

    def test_shelves_and_window(self):
        w1 = work(1)
        self.assertEqual(w1["want_to_read"], 3)
        self.assertEqual(w1["already_read"], 1)
        # window (2021-09-30, 2022-09-30] holds Sep 2022 and Dec 2021
        self.assertEqual(w1["wtr_last_12m"], 2)
        self.assertEqual(w1["wtr_prev_12m"], 0)
        self.assertEqual(work(2)["currently_reading"], 1)

    def test_ratings(self):
        self.assertEqual(work(1)["ratings_count"], 2)
        self.assertAlmostEqual(work(1)["ratings_mean"], 4.5)
        self.assertEqual(work(3)["ratings_count"], 0)   # 9 is invalid

    def test_supply_counts(self):
        w1 = work(1)
        self.assertEqual(w1["n_editions"], 2)
        self.assertEqual(w1["n_recent_editions"], 1)
        self.assertEqual(w1["n_publishers"], 1)      # "Old Press" twice
        self.assertEqual(w1["n_recent_publishers"], 1)
        self.assertEqual(w1["n_digitized"], 1)
        self.assertEqual(w1["n_filipino_editions"], 1)
        self.assertEqual(work(3)["n_filipino_editions"], 1)  # fil code
        self.assertEqual(work(3)["n_publishers"], 2)
        self.assertEqual(work(5)["n_editions"], 0)   # zero, not missing


class TestWatchlist(unittest.TestCase):

    def test_watchlist_selection(self):
        listed = helper.build_watchlist(profile, min_want_to_read=1,
                                        max_recent_editions=1, top_n=10)
        # W2 also enters 2027 but nobody wants to read it yet
        self.assertEqual(listed["work_key"].tolist(), ["/works/OL1W"])

    def test_author_names_in_credit_order(self):
        listed = helper.build_pd_list(profile, min_want_to_read=0,
                                      top_n=10, authors=tables["authors"])
        names = dict(zip(listed["work_key"], listed["authors"]))
        self.assertEqual(names["/works/OL3W"], "Mid Century")
        listed = helper.build_watchlist(profile, 0, 5, top_n=10,
                                        authors=tables["authors"])
        names = dict(zip(listed["work_key"], listed["authors"]))
        self.assertEqual(names["/works/OL2W"],
                         "Died Seventy-Six, Mid Century")

    def test_low_supply_cutoff(self):
        listed = helper.build_watchlist(profile, min_want_to_read=1,
                                        max_recent_editions=0, top_n=10)
        self.assertTrue(listed.empty)   # W1 has one recent edition

    def test_pd_list(self):
        listed = helper.build_pd_list(profile, min_want_to_read=1, top_n=10)
        self.assertEqual(listed["work_key"].tolist(), ["/works/OL3W"])

    def test_low_supply_pd_list(self):
        listed = helper.build_pd_list(profile, min_want_to_read=1, top_n=10,
                                      max_recent_editions=0)
        self.assertTrue(listed.empty)   # W3 has one edition from 2020

    def test_top_authors(self):
        table = helper.top_authors(profile, tables["authors"],
                                   "enters_2027", top_n=10)
        first = table.iloc[0]
        self.assertEqual((first["name"], first["works"],
                          first["want_to_read"], first["via_us_1931"]),
                         ("Living Or Unknown", 1, 3, 1))
        # W2 counts once for each of its two authors, both via PH 1976
        rest = table[table["name"] != "Living Or Unknown"]
        self.assertEqual(sorted(rest["name"]),
                         ["Died Seventy-Six", "Mid Century"])
        self.assertEqual(rest["via_ph_1976"].tolist(), [1, 1])

    def test_group_comparison(self):
        groups = helper.compare_groups(profile).set_index("us_code")
        self.assertEqual(groups["works"].sum(), 7)
        self.assertEqual(groups.loc["enters", "total_wtr"], 3)
        self.assertEqual(groups.loc["enters", "wtr_last_12m"], 2)
        # no entries in the earlier window, so growth is undefined
        self.assertTrue(pd.isna(groups.loc["enters", "wtr_growth_pct"]))
        self.assertAlmostEqual(groups.loc["enters", "mean_rating"], 4.5)

    def test_flag_summary(self):
        table = helper.list_flag_summary(profile, 1).set_index("list")
        self.assertEqual(
            table.loc["potentially public domain, high demand", "works"], 1)
        self.assertEqual(table.loc["enters 2027, high demand", "works"], 1)
        self.assertEqual(table["gap_after_death"].sum(), 0)

    def test_threshold_ignores_unshelved_works(self):
        # WTR counts among works with any entry are [3, 1]
        self.assertEqual(helper.demand_threshold(profile, 0.99), 3)

    def test_validation_checks_pass(self):
        listed = helper.build_watchlist(profile, 1, 1, top_n=10)
        checks = helper.validate_analysis(tables, status, profile, listed,
                                          min_want_to_read=1)
        failed = checks[~checks["passed"]]
        self.assertTrue(failed.empty, msg=failed.to_string())

## 4 Code style

`pycodestyle` checks line length, spacing and other PEP 8 conventions,
so the module stays easy to read for the peer reviewers.

In [6]:
class TestStyle(unittest.TestCase):

    def test_pep8(self):
        import pycodestyle

        report = pycodestyle.StyleGuide(quiet=True).check_files(
            ["helper_code.py"])
        self.assertEqual(report.total_errors, 0)

## 5 Regime table, hand-checked audit and diagram

These tests cover the functions added for the methodology section. The audit test feeds in "checked" facts that we know disagree with the fixture in one place (W1's year), so the function must report exactly that row as wrong, while a row with nothing checkable must be reported as not verifiable rather than as correct.

In [7]:
class TestAuditAndRegimes(unittest.TestCase):

    def test_compare_codes(self):
        self.assertEqual(helper.compare_codes("pd", None), "not checked")
        self.assertEqual(helper.compare_codes("unknown_death", "pd"),
                         "missing in data")
        self.assertEqual(helper.compare_codes("pd", "pd"), "correct")
        self.assertEqual(helper.compare_codes("enters", "renewal"), "wrong")

    def test_audit_flags_the_planted_error(self):
        sample = [
            # W1 is catalogued as 1931; pretend the true year is 1932
            ("W", "/works/OL1W", 1932, None, "planted year error"),
            # W3 matches: editions from 1890, author died 1950
            ("P", "/works/OL3W", 1890, 1950, "should match"),
            # nothing checked for W5
            ("P", "/works/OL5W", None, None, "nothing to check"),
        ]
        audit = helper.audit_estimates(profile, sample).set_index("title")
        w1 = audit.loc["Book Nineteen Thirty-One"]
        self.assertEqual((w1["us_outcome"], w1["checked_us"], w1["verdict"]),
                         ("wrong", "renewal", "wrong"))
        self.assertEqual(w1["ph_outcome"], "not checked")
        w3 = audit.loc["Old Classic"]
        self.assertEqual((w3["us_outcome"], w3["ph_outcome"], w3["verdict"]),
                         ("correct", "correct", "correct"))
        self.assertEqual(audit.loc["No Author", "verdict"], "not verifiable")
        summary = helper.audit_summary(audit.reset_index())
        us_rows = summary[summary["regime"] == "United States"]
        self.assertEqual(int(us_rows["wrong"].sum()), 1)
        self.assertEqual(int(us_rows["correct"].sum()), 1)

    def test_regime_table_counts_every_work(self):
        table = helper.status_by_regime(status)
        # each regime column classifies all 7 fixture works exactly once
        self.assertEqual(table.sum().tolist(), [7, 7, 7])
        # W3's author died in 1950, so it is free under life + 70 too
        self.assertGreaterEqual(
            table.loc["Public domain already", "Life + 70 (comparison)"], 1)

    def test_selection_funnel(self):
        funnel = helper.selection_funnel(profile, 1, 1).set_index(
            ["list", "step"])
        watch = funnel.loc["2027 watchlist", "works"].tolist()
        # 7 works -> 6 unflagged (W6) -> W1, W2 enter -> only W1 is wanted
        # -> W1 has one recent edition, so it survives the supply filter
        self.assertEqual(watch, [7, 6, 2, 1, 1])
        free = funnel.loc["public-domain list", "works"].tolist()
        # W3 and W4 are free; only W3 has a Want-to-Read entry
        self.assertEqual(free, [7, 6, 2, 1])
        self.assertAlmostEqual(
            funnel.loc[("2027 watchlist", "no data-quality flag"),
                       "pct_of_previous"], round(100 * 6 / 7, 2))

    def test_report_style_and_css(self):
        helper.use_report_style()
        self.assertEqual(plt.rcParams["axes.facecolor"].lower(),
                         helper.PAPER.lower())
        self.assertIn(helper.INK, helper.notebook_css())

    def test_pipeline_diagram_draws(self):
        fig = helper.plot_pipeline_diagram()
        self.assertGreater(len(fig.axes[0].patches), 10)
        plt.close(fig)

## 6 Locating the dumps on Jojie

The report reads one fixed folder, `/mnt/data/public/openlibrary/20221007` (the only snapshot there with separate works, editions, authors, reading-log and ratings dumps). These tests check that this is the default folder, that a missing folder is an error rather than a silent fallback, that discovery accepts dated, `latest` and uncompressed files, that the optional Wikidata dump may be absent, and that the date check recognises the snapshot's `latest` files as the September 30, 2022 dump.


In [8]:
class TestDumpDiscovery(unittest.TestCase):

    def make_files(self, folder, names):
        folder.mkdir(parents=True, exist_ok=True)
        for name in names:
            (folder / name).write_text("")

    def test_dated_beats_latest_and_gz_beats_txt(self):
        with tempfile.TemporaryDirectory() as directory:
            folder = Path(directory)
            self.make_files(folder, [
                "ol_dump_works_2026-09-30.txt.gz",
                "ol_dump_works_2026-10-31.txt",
                "ol_dump_works_2026-10-31.txt.gz",
                "ol_dump_works_latest.txt.gz",
                "ol_dump_ratings_latest.txt.gz",
                "ol_dump_editions_2026-10-31.txt",
            ])
            found = helper.discover_dumps(
                folder, names=("works", "ratings", "editions"))
            self.assertEqual(found["works"].name,
                             "ol_dump_works_2026-10-31.txt.gz")
            self.assertEqual(found["ratings"].name,
                             "ol_dump_ratings_latest.txt.gz")
            self.assertEqual(found["editions"].name,
                             "ol_dump_editions_2026-10-31.txt")

    def test_fixed_jojie_folder(self):
        self.assertEqual(
            Path(helper.DATA_ROOT) / helper.DATA_SNAPSHOT,
            Path("/mnt/data/public/openlibrary/20221007"))
        with tempfile.TemporaryDirectory() as directory:
            root = Path(directory)
            (root / helper.DATA_SNAPSHOT).mkdir()
            self.assertEqual(helper.resolve_data_dir(root=root),
                             root / helper.DATA_SNAPSHOT)
            self.assertEqual(helper.resolve_data_dir(custom_dir=root), root)
            # no fallback search: a missing folder is an error
            with self.assertRaises(FileNotFoundError):
                helper.resolve_data_dir(root=root, snapshot="19990101")

    def test_wikidata_is_optional(self):
        with tempfile.TemporaryDirectory() as directory:
            folder = Path(directory)
            self.make_files(folder, [
                f"ol_dump_{name}_latest.txt.gz"
                for name in helper.DUMP_NAMES])
            found = helper.discover_dumps(folder)
            self.assertEqual(set(found), set(helper.DUMP_NAMES))
            (folder / "ol_dump_ratings_latest.txt.gz").unlink()
            with self.assertRaises(FileNotFoundError):
                helper.discover_dumps(folder)

    def test_dump_date_check(self):
        snapshot = Path(helper.DATA_ROOT) / helper.DATA_SNAPSHOT
        latest = {"works": snapshot / "ol_dump_works_latest.txt.gz"}
        self.assertEqual(helper.dump_date_of(latest["works"]), "2022-09-30")
        self.assertTrue(helper.check_dump_date(latest).startswith("All"))
        same = {"works": Path("ol_dump_works_2022-09-30.txt.gz")}
        self.assertTrue(helper.check_dump_date(same).startswith("All"))
        other = {"works": Path("/x/20260424/ol_dump_works_latest.txt.gz")}
        message = helper.check_dump_date(other)
        self.assertTrue(message.startswith("WARNING"))
        self.assertIn("2026-04-24", message)


## 7 Whole-snapshot files: revision history and all-types dump

The revision fixture gives author OL1A three revisions whose death year moves from none to 1975 to 1976, so its Philippine status changes from public domain to "enters 2027"; OL2A has one stable value, OL3A none, and a deleted author OL9A must be ignored. The all-types fixture reuses the record lines above, so its live counts must equal the parsed tables. The manifest and lock tests check that cached tables built under other rules are refused and that a lock left by a dead process is taken over.


In [9]:
def rev_line(record_type, key, revision, day, record):
    """One line of the revision (cdump) format."""
    record = {"key": key, "type": {"key": record_type}, **record}
    return "\t".join([record_type, key, str(revision), day + "T00:00:00",
                      json.dumps(record)])


revision_lines = [
    rev_line("/type/author", "/authors/OL1A", 1, "2009-05-01", {"name": "A"}),
    rev_line("/type/author", "/authors/OL1A", 2, "2015-03-01",
             {"name": "A", "death_date": "1975"}),
    rev_line("/type/author", "/authors/OL1A", 3, "2020-01-01",
             {"name": "A", "death_date": "1976"}),
    rev_line("/type/author", "/authors/OL2A", 1, "2010-01-01",
             {"name": "B", "death_date": "1951"}),
    rev_line("/type/author", "/authors/OL3A", 1, "2011-01-01", {"name": "C"}),
    rev_line("/type/author", "/authors/OL9A", 1, "2012-01-01",
             {"name": "Deleted", "death_date": "1976"}),
    rev_line("/type/work", "/works/OL1W", 1, "2012-01-01", {"title": "x"}),
    rev_line("/type/work", "/works/OL1W", 2, "2013-01-01", {"title": "y"}),
]
history_dir = Path(tmp.name) / "history"
history_dir.mkdir()
with gzip.open(history_dir / "ol_cdump_latest.txt.gz", "wt") as handle:
    handle.write("\n".join(revision_lines) + "\n")
with gzip.open(history_dir / "ol_dump_latest.txt.gz", "wt") as handle:
    handle.write("\n".join(works + editions + authors) + "\n")
history_paths = helper.build_history_tables(spark, history_dir,
                                            Path(tmp.name) / "hist_pq")
revisions_fx = spark.read.parquet(str(history_paths["revisions"]))
full_fx = spark.read.parquet(str(history_paths["full_dump"]))


class TestHistory(unittest.TestCase):

    def test_revision_rows(self):
        self.assertEqual(revisions_fx.count(), len(revision_lines))
        # keys are kept for authors only
        self.assertEqual(revisions_fx.filter("key is not null").count(), 6)

    def test_death_year_history(self):
        result = helper.death_year_history(revisions_fx,
                                           deaths["author_deaths"])
        row = result["stability"].iloc[0]
        # OL9A is not a current author, so 3 authors remain
        self.assertEqual(int(row["authors"]), 3)
        self.assertEqual(int(row["death_year_changed"]), 1)    # OL1A
        self.assertEqual(int(row["ph_status_changed"]), 1)     # pd -> enters
        a1 = result["per_author"].filter(
            "author_key = '/authors/OL1A'").collect()[0]
        self.assertEqual(a1["first_death_year"], 1975)
        self.assertEqual(str(a1["death_added"]), "2015-03-01")
        entrants = result["entrants"]
        self.assertEqual(entrants["year_recorded"].tolist(), [2015])

    def test_revision_activity(self):
        activity = helper.revision_activity(revisions_fx)
        self.assertEqual(int(activity.loc[2013, "edits_works"]), 1)
        self.assertEqual(int(activity.loc[2009, "created_authors"]), 1)

    def test_full_dump_census_matches_tables(self):
        census = helper.full_dump_census(full_fx, tables).set_index(
            "namespace")
        self.assertTrue(census["matches"].all())
        self.assertEqual(int(census.loc["/works", "redirect"]), 1)

    def test_manifest_refuses_other_rules(self):
        folder = Path(tmp.name) / "manifest_test"
        helper.check_manifest(folder, dumps)
        saved = json.loads((folder / helper.MANIFEST).read_text())
        saved["recent_from_year"] = 2017
        (folder / helper.MANIFEST).write_text(json.dumps(saved))
        with self.assertRaises(RuntimeError):
            helper.check_manifest(folder, dumps)

    def test_stale_lock_is_taken_over(self):
        lock = Path(tmp.name) / "stale.lock"
        import socket
        lock.write_text(f"{socket.gethostname()} 999999999")
        helper.acquire_lock(lock)          # dead PID: taken over
        self.assertIn(str(os.getpid()), lock.read_text())
        with self.assertRaises(RuntimeError):
            helper.acquire_lock(lock)      # now held by a live process


## 8 Exploration, missing-value and snapshot helpers

`describe_raw_data` must report the fixture's own statistics, which we can count by hand: six reading-log entries (the custom shelf is dropped), four of them Want to Read (66.7%), and two valid ratings with a mean of 4.50 (the rating of 9 is out of range). `investigate_missing` must account for every author, work and edition exactly once. `snapshot_inventory` must mark a folder with all five record dumps as complete and a folder holding only the revision history as incomplete, which is how the report justifies its choice of snapshot.

In [10]:
class TestNewHelpers(unittest.TestCase):

    def test_describe_raw_data(self):
        stats = helper.describe_raw_data(
            {**tables, "revisions": revisions_fx, "full_dump": full_fx})
        value = stats.set_index(["dataset", "statistic"])["value"]
        self.assertEqual(value["reading-log", "shelf entries"], "6")
        self.assertEqual(value["reading-log", "Want to Read share"], "66.7%")
        self.assertEqual(value["reading-log", "date range"],
                         "2020-06-01 to 2022-09-01")
        self.assertEqual(value["ratings", "mean stars (1-5)"], "4.50")
        self.assertEqual(value["works", "records"], "7")
        self.assertEqual(value["revision history", "revisions (all types)"],
                         f"{len(revision_lines):,}")

    def test_investigate_missing_counts_every_row_once(self):
        result = helper.investigate_missing(tables)
        n_authors = tables["authors"].count()
        no_death = tables["authors"].filter("ol_death_year is null").count()
        self.assertEqual(int(result["authors"]["authors"].sum()), n_authors)
        self.assertEqual(int(result["authors"]["without_death_year"].sum()),
                         no_death)
        self.assertEqual(int(result["works"]["works"].sum()), 7)
        self.assertEqual(int(result["editions"]["editions"].sum()),
                         tables["editions"].count())

    def test_snapshot_inventory(self):
        with tempfile.TemporaryDirectory() as directory:
            root = Path(directory)
            full = root / "20221007"
            full.mkdir()
            for name in helper.DUMP_NAMES:
                (full / f"ol_dump_{name}_latest.txt.gz").write_text("")
            (root / "20260424").mkdir()
            (root / "20260424" / "ol_cdump_latest.txt.gz").write_text("")
            table = helper.snapshot_inventory(root).set_index("snapshot")
            needed = list(helper.DUMP_NAMES)
            self.assertTrue(table.loc["20221007", needed].all())
            self.assertFalse(table.loc["20260424", needed].any())

    def test_review_watchlist(self):
        listed = pd.DataFrame({"work_key": ["/works/OL273188W", "/works/X"]})
        table = helper.review_watchlist(listed)
        self.assertEqual(table["check"].tolist(),
                         ["false positive", "not checked"])
        self.assertIn("Dodd", table.loc[0, "note"])


## 9 Merged records: redirects and duplicate titles

Two helpers repair links that would otherwise be lost. `resolve_redirect_chains` must follow a chain of merges to its end (A was merged into B, and B later into C, so A and B both end at C) and must drop a redirect that points to itself or runs in a loop. `resolve_merged_keys` must then move a shelf entry logged against the merged work OL8W onto the surviving work OL1W, and replace a merged author key inside a work's author list without duplicating the author.

`consolidate_titles` merges work records with the same normalized title and the same authors. The test adds a second record for work 1 (title in other capitals and punctuation, two Want-to-Read entries, first year 1950, three recent editions) and a second record for the authorless work 5. By hand: the two records of work 1 become one title with 3 + 2 = 5 entries, 1 + 3 = 4 recent editions and the earlier first year (1931, so it still enters in 2027), while the two authorless records stay apart because nothing shows they are the same book. Nine records therefore give eight titles.

`first_year_sensitivity` is checked on three hand-made works: X has edition years 1900, 1950 and 1951 (the cautious rule moves it from 1900 to 1950, out of the public-domain group), Z has its own first-publish year 1899 (which still counts, so it stays), and Y has only two distinct years (not tested). A fourth work with a single edition year checks that a work with nothing to compare is left alone.

`born_long_ago_sensitivity` is checked on four works that share an author born in 1840 with no death year: the work moves to public domain when its other authors died before 1976 (W1, W2), and stays unknown when a co-author may be alive (W3) or died in 1980 (W4).


In [11]:
from pyspark.sql import types as T

redirects_fx = spark.read.parquet(str(history_paths["redirects"]))


class TestRedirects(unittest.TestCase):

    def test_parse_redirects(self):
        rows = [(r["old_key"], r["new_key"]) for r in redirects_fx.collect()]
        self.assertEqual(rows, [("/works/OL8W", "/works/OL1W")])

    def test_chains_loops_and_self_redirects(self):
        raw = spark.createDataFrame(
            [("A", "B"), ("B", "C"),      # chain: both end at C
             ("D", "D"),                  # points to itself
             ("X", "Y"), ("Y", "X")],     # loop
            ["old_key", "new_key"])
        resolved = {r["old_key"]: r["new_key"] for r in
                    helper.resolve_redirect_chains(raw).collect()}
        self.assertEqual(resolved, {"A": "C", "B": "C"})

    def test_resolve_merged_keys(self):
        redirects = spark.createDataFrame(
            [("/works/OL8W", "/works/OL1W"),
             ("/authors/OL9A", "/authors/OL1A")], ["old_key", "new_key"])
        log = spark.createDataFrame(
            [("/works/OL8W", "Want to Read"), ("/works/OL3W", "Want to Read")],
            ["work_key", "shelf"])
        works_fx = spark.createDataFrame(
            [("/works/OL1W", ["/authors/OL1A", "/authors/OL9A"]),
             ("/works/OL2W", ["/authors/OL2A"])],
            ["work_key", "author_keys"])
        fixed = helper.resolve_merged_keys(
            {"works": works_fx, "editions": log, "reading_log": log,
             "ratings": log}, redirects)
        self.assertEqual(
            sorted(r["work_key"] for r in fixed["reading_log"].collect()),
            ["/works/OL1W", "/works/OL3W"])
        authors_of = {r["work_key"]: r["author_keys"]
                      for r in fixed["works"].collect()}
        # the merged key is replaced and the author is not listed twice
        self.assertEqual(authors_of["/works/OL1W"], ["/authors/OL1A"])
        self.assertEqual(authors_of["/works/OL2W"], ["/authors/OL2A"])

    def test_redirect_impact(self):
        redirects = helper.resolve_redirect_chains(redirects_fx)
        log = spark.createDataFrame(
            [("/works/OL8W",), ("/works/OL1W",), ("/works/OL999W",)],
            ["work_key"])
        raw = {**tables, "reading_log": log}
        impact = helper.redirect_impact(
            raw, helper.resolve_merged_keys(raw, redirects)).set_index("link")
        row = impact.loc["reading-log entry -> work"]
        # OL1W matches as is, OL8W after resolving, OL999W never
        self.assertEqual((row["rows"], row["matched_raw"], row["recovered"],
                          row["still_unmatched"]), (3, 1, 1, 1))


class TestTitles(unittest.TestCase):

    def test_normalize_title(self):
        cases = {
            "The Intelligent Investor": "intelligent investor",
            "Intelligent investor.": "intelligent investor",
            "A Room of One's Own": "room of one s own",
            "Преступление и наказание": "преступление и наказание",
            "Theatre": "theatre",          # not a leading article
            "": "", None: "",
        }
        for text, expected in cases.items():
            self.assertEqual(helper.normalize_title(text), expected,
                             msg=text)

    def test_spark_twin(self):
        texts = ["The Intelligent Investor", "Intelligent investor.",
                 "A Room of One's Own", "Преступление и наказание",
                 "Theatre", "", None]
        df = spark.createDataFrame([(t,) for t in texts],
                                   T.StructType([T.StructField(
                                       "t", T.StringType())]))
        got = [r["k"] for r in df.select(
            helper.title_key(F.col("t")).alias("k")).collect()]
        self.assertEqual(got, [helper.normalize_title(t) for t in texts])

    def test_consolidate_titles(self):
        def copy_of(key, new_key, **changes):
            """A second record for one fixture work, with some changes."""
            row = profile.filter(F.col("work_key") == key)
            for column, value in changes.items():
                row = row.withColumn(
                    column, F.lit(value).cast(profile.schema[column]
                                              .dataType))
            return row.withColumn("work_key", F.lit(new_key))

        records = (
            profile
            .unionByName(copy_of(
                "/works/OL1W", "/works/OL91W",
                title="the BOOK nineteen thirty-one!", first_year=1950,
                want_to_read=2, n_recent_editions=3))
            .unionByName(copy_of("/works/OL5W", "/works/OL95W"))
        )
        titles = helper.consolidate_titles(records)
        self.assertEqual(records.count(), 9)
        self.assertEqual(titles.count(), 8)
        merged = titles.filter("n_records > 1").collect()
        self.assertEqual(len(merged), 1)
        row = merged[0]
        self.assertEqual(row["work_key"], "/works/OL1W")   # most shelved
        self.assertEqual(row["want_to_read"], 5)
        self.assertEqual(row["n_recent_editions"], 4)
        self.assertEqual((row["first_year"], row["us_code"]),
                         (1931, "enters"))
        self.assertTrue(row["enters_2027"])
        summary = helper.consolidation_summary(records, titles).iloc[0]
        self.assertEqual(int(summary["records_absorbed"]), 1)
        # the merged title holds 5 of the 6 Want-to-Read entries
        self.assertAlmostEqual(summary["wtr_on_merged_titles_pct"], 83.3)
        # list builders accept the title table and show n_records
        listed = helper.build_watchlist(titles, 1, 5, top_n=5)
        self.assertEqual(listed.loc[0, "n_records"], 2)

    def test_first_year_sensitivity(self):
        editions_fx = spark.createDataFrame(
            [("X", 1900), ("X", 1950), ("X", 1951),
             ("Y", 1931), ("Y", 1931), ("Y", 1940),
             ("Z", 1899), ("Z", 1960), ("Z", 1970),
             ("V", 1925)],                # a single year must not fail
            ["work_key", "publish_year"])
        status_fx = spark.createDataFrame(
            [("X", None, 1900, "pd"), ("Y", None, 1931, "enters"),
             ("Z", 1899, 1899, "pd"), ("V", None, 1925, "pd")],
            "work_key string, work_year int, first_year int, us_code string")
        table = helper.first_year_sensitivity(
            editions_fx, status_fx).set_index("US status")
        free = table.loc[helper.US_LABELS["pd"]]
        self.assertEqual((free["works"], free["tested"], free["moved"]),
                         (3, 2, 1))
        entering = table.loc[helper.US_LABELS["enters"]]
        self.assertEqual((entering["tested"], entering["moved"]), (0, 0))

    def test_born_long_ago_sensitivity(self):
        status_fx = spark.createDataFrame(
            [("W1", "renewal", ["old", "dead"]),   # both certainly dead
             ("W2", "pd", ["old"]),                # moves, already US PD
             ("W3", "renewal", ["old", "recent"]),  # one may be alive
             ("W4", "renewal", ["old", "late"])],   # co-author died 1980
            ["work_key", "us_code", "author_keys"]
        ).withColumn("ph_code", F.lit("unknown_death"))
        deaths_fx = spark.createDataFrame(
            [("old", None, 1840), ("dead", 1950, 1880),
             ("recent", None, 1950), ("late", 1980, 1900)],
            "author_key string, death_year int, birth_year int")
        row = helper.born_long_ago_sensitivity(status_fx, deaths_fx).iloc[0]
        self.assertEqual(int(row["works_unknown_death"]), 4)
        self.assertEqual(int(row["would_become_ph_public_domain"]), 2)
        self.assertEqual(int(row["of_which_not_us_public_domain"]), 1)


## 10 Run every test

In [12]:
suite = unittest.TestSuite()
loader = unittest.defaultTestLoader
for case in [TestYearParsing, TestPublisherNames, TestCopyrightRules,
             TestSparkTwins, TestParsing, TestAuthorDeaths, TestStatus,
             TestDemandAndSupply, TestWatchlist, TestAuditAndRegimes,
             TestDumpDiscovery, TestHistory, TestNewHelpers, TestRedirects,
             TestTitles, TestStyle]:
    suite.addTests(loader.loadTestsFromTestCase(case))
# write the per-test log to stdout so it is kept with the notebook
result = unittest.TextTestRunner(stream=sys.stdout, verbosity=2).run(suite)
print(f"\n{result.testsRun - len(result.failures) - len(result.errors)}"
      f" of {result.testsRun} tests passed")
tmp.cleanup()
assert result.wasSuccessful(), "Some tests failed; see the log above."

test_life_years (__main__.TestYearParsing.test_life_years) ... 

ok


test_publication_years (__main__.TestYearParsing.test_publication_years) ... 

ok


test_wikidata_time (__main__.TestYearParsing.test_wikidata_time) ... 

ok


... (138 outputs left out)

test_pep8 (__main__.TestStyle.test_pep8) ... 

ok


----------------------------------------------------------------------
Ran 73 tests in 104.300s

OK



73 of 73 tests passed
